## Compute independent cluster probabilities.
In this notebook we estimate the probability of each cluster as independent trials. Let $C_k$ be the event that cluster $k$ is present in the sample. Applying the law of total probability we obtain
$$
P(C_k = 1) = \sum_\alpha P(C_k = 1\, |\, I = \alpha)P(I=\alpha)
$$
where $I$ is the initial state. 

In [ ]:
import sqlite3
import pandas as pd
import os
import numpy as np
import ast
import matplotlib.pyplot as plt

# Set the display format for pandas DataFrames to scientific notation
pd.set_option('display.float_format', '{:.6e}'.format)

# Change to the src directory
# This is necessary to ensure that the script runs in the correct context
# and can find the necessary modules and files.
os.chdir("/home/ebr/projects/release-volume-sampler/src")
rundir = "/home/ebr/projects/release-volume-sampler/generated/messina_20250806"
db_path = os.path.join(rundir, "volumes/volumes.db")

In [ ]:
# Load seed triangle probabilities from the database
db_path = os.path.join(rundir, "volumes/volumes.db")
with sqlite3.connect(db_path) as conn:
    df_seed_triangles = pd.read_sql_query(f"SELECT * FROM seed_triangles", conn)

# Get list of clusters from the database
with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT DISTINCT cluster FROM volumes")
    clusters = sorted([row[0] for row in cursor.fetchall()])
print(clusters)

# Get all p_shake columns
p_shake_cols = [col for col in df_seed_triangles.columns if col.startswith('p_shake')]

# Prepare a DataFrame to store results
result = pd.DataFrame(index=clusters, columns=p_shake_cols)


In [ ]:
def canonicalize_seed_triangles(val):
    # Convert string to tuple, or leave as is if already tuple
    if isinstance(val, str):
        return tuple(ast.literal_eval(val))
    return tuple(val)

def get_volumes_for_cluster(db_path, cluster_id):
    """
    Fetch all volumes mapped to a given cluster and return as a DataFrame.
    """
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query(
            "SELECT * FROM volumes WHERE cluster = ?", conn, params=(cluster_id,)
        )
    return df

# For each cluster, compute the probability for each p_shake column
for cluster in clusters:
    print(cluster)
    # Load volumes for the current cluster
    df_cluster = get_volumes_for_cluster(db_path, cluster)
    
    # Group by the canonicalized seed triangles and sum condprob for fixed seed triangles
    df_cluster['seed_triangles_tuple'] = df_cluster['seed_triangles'].apply(canonicalize_seed_triangles)
    condprob_sum = df_cluster.groupby('seed_triangles_tuple')['condprob'].sum().reset_index()
    
    # For each p_shake column, compute the probability of the cluster
    for pcol in p_shake_cols:
        # Map triangle_id to p_shake for this column
        p_shake_map = dict(zip(df_seed_triangles['triangle_id'], df_seed_triangles[pcol]))
        # Compute p_row for each tuple in condprob_sum for this cluster
        p_row = condprob_sum['condprob'] * condprob_sum['seed_triangles_tuple'].apply(
            lambda tup: np.prod([p_shake_map.get(tri, 1.0) for tri in tup])
        )
        # Probability at least one occurs
        cluster_prob = 1 - np.prod(1 - p_row)
        print(f"Cluster {cluster}, {pcol}: {cluster_prob:.3e}")
        result.loc[cluster, pcol] = cluster_prob

# 4. Convert to float for display
result = result.astype(float)
print(result)

In [ ]:
result.drop(columns=['p_shake'], inplace=True, errors='ignore')

In [ ]:
result

In [ ]:
# Sort columns numerically from p_shake_0 to p_shake_120
def p_shake_key(col):
    try:
        return int(col.split('_')[-1])
    except Exception:
        return float('inf')

sorted_cols = sorted([col for col in result.columns if col.startswith('p_shake')], key=p_shake_key)
result = result[sorted_cols]

In [ ]:
# write result to file.
aggregation_dir = os.path.join(rundir, "aggregation")
os.makedirs(aggregation_dir, exist_ok=True)
np.savez(os.path.join(aggregation_dir, "cluster_release_probabilities.npz"), result=result)

In [ ]:
# Create a heatmap of the probabilities

plt.figure(figsize=(10, 6))
log_result = np.log10(result.values + 1e-20)
im = plt.imshow(log_result, aspect='auto', cmap='coolwarm', vmin=-3, vmax=0)
plt.colorbar(im, label='log10(Probability)')

# Set ticks every 10 clusters (rows)
yticks = np.arange(0, len(result.index), 10)
plt.yticks(ticks=yticks, labels=[result.index[i] for i in yticks])

# Set ticks every 10 shakemaps (columns)
xticks = np.arange(0, len(result.columns), 10)
plt.xticks(ticks=xticks, labels=[result.columns[i] for i in xticks], rotation=90)

plt.title("Cluster probabilities (log10 scale)")
plt.tight_layout()

plt.savefig(os.path.join(aggregation_dir, "cluster_release_probabilities.png"))
plt.show()

## Load data and compute exceedance curves

We are interested in computing exceedance curves of MIH. To this end we assume that we can consider each cluster as independent and that there is no interaction. Let S_i refer to the scenario associated with the shakemaps. According to the law of total probability
$$
P(MIH > h) = \sum_i P(MIH > h|S_i)P(S_i).
$$
Assuming no wave interaction $MIH = \max_k MIH_k$ where $MIH_k$ is the maximal inundation height associated with each cluster. Applying indpendence it follows that
$$
P(MIH > h \,|\, S_i) = 1 - \prod_k P(MIH_k < h \,|\, S_i)
$$
In this notebook we will apply no2d instead as that is a value we allready have available for the clusters.

In [ ]:
import numpy as np

# Load the saved result
data = np.load(os.path.join(rundir, "aggregation", "cluster_release_probabilities.npz"), allow_pickle=True)
cluster_release_probabilities = pd.DataFrame(data['result'])

# Load the cluster no2d value from database
with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT cluster, no2d FROM volumes where is_representative = 1 order by cluster")
    cluster_no2d = pd.DataFrame(cursor.fetchall(), columns=['cluster', 'no2d'])

# Assume scenarios are equally weighted
weights = np.ones(len(cluster_release_probabilities.columns))/len(cluster_release_probabilities.columns)

In [ ]:
cluster_no2d

In [ ]:
weights.sum()

In [ ]:
cluster_release_probabilities

In [ ]:
def exceedance_probability(cluster_release_probabilities, cluster_no2d, threshold, weights):
    """
    Calculate the exceedance probability for clusters with no2d > threshold.
    """
    # Filter clusters with no2d > threshold
    filtered_probabilities = cluster_release_probabilities[cluster_no2d.no2d >= threshold]
    
    # Calculate the exceedance probability
    exceedance_probability_by_scenario = 1 - (1 - filtered_probabilities).prod(axis=0)
    
    # Weight the exceedance probabilities
    weighted_exceedance_probability = np.dot(exceedance_probability_by_scenario, weights)
    return weighted_exceedance_probability

for threshold in np.linspace(0, 5, 10):
    exceedance_prob = exceedance_probability(cluster_release_probabilities, cluster_no2d, threshold, weights)
    print(f"Exceedance probability for no2d > {threshold:.2f}:")
    print(exceedance_prob)

In [ ]:
import numpy as np

def exceedance_probability(probs, value, thresholds, weights):
    """
    Vectorized calculation of exceedance probabilities for an array of thresholds.
    Returns a numpy array of weighted exceedance probabilities for each threshold.

    probs:  2D numpy array of shape (num_clusters, num_scenarios) Probability of release by cluster and scenario.
    value: 1D numpy array of shape (num_clusters,) Value to be exceeded by cluster.
    thresholds: 1D numpy array of shape (num_thresholds,) Thresholds for exceedance.
    weights: 1D numpy array of shape (num_scenarios,) Weights for each scenario.
    """
    # Create a mask matrix: shape (num_thresholds, num_clusters)
    mask = value[None, :] >= thresholds[:, None]  # shape (T, C)

    # For each threshold, mask out clusters not exceeding the threshold
    # Set probabilities for clusters not exceeding threshold to 0 (so they don't affect the product)
    masked_probs = np.where(mask[:, :, None], probs[None, :, :], 0.0)  # shape (T, C, S)

    # Compute product over clusters (axis=1), for each threshold and scenario
    prod = np.prod(1 - masked_probs, axis=1)  # shape (T, S)
    exceed_by_scenario = 1 - prod  # shape (T, S)

    # Weighted sum over scenarios for each threshold
    weighted = np.dot(exceed_by_scenario, weights)  # shape (T,)

    return weighted

# Example usage:
thresholds = np.linspace(0, 5, 10)
exceedance_probs = exceedance_probability(cluster_release_probabilities.values, cluster_no2d['no2d'].values, thresholds, weights)
for t, p in zip(thresholds, exceedance_probs):
    print(f"Exceedance probability for no2d > {t:.2f}: {p:.6e}")

In [ ]:
thresholds = np.linspace(0, 5, 100)
exceedance_probs = exceedance_probability(cluster_release_probabilities.values, cluster_no2d['no2d'].values, thresholds, weights)

plt.figure(figsize=(10, 6))
plt.plot(thresholds, exceedance_probs, marker='o', markersize=3)
plt.title("Exceedance Probability vs NO2D Threshold")
plt.xlabel("NO2D Threshold")
plt.ylabel("Exceedance Probability")
plt.grid()
plt.show()


Note that the exceedance probability does not tend to 1 as threshold tends to 0. This is simply because there is a probability than none of the clusters are activated.